In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


In [ ]:
def get_intersections(ifc_file, dataframe=True, figure=True):
    from topologicpy.Vertex import Vertex
    from topologicpy.Cluster import Cluster
    from topologicpy.Topology import Topology
    from topologicpy.Dictionary import Dictionary
    from topologicpy.Plotly import Plotly
    import numpy as np
    import pandas as pd

    def bb(topology):
        vertices = Topology.Vertices(_as_topology(topology))
        x = []
        y = []
        z = []
        for aVertex in vertices:
            x.append(Vertex.X(_as_topology(aVertex)))
            y.append(Vertex.Y(_as_topology(aVertex)))
            z.append(Vertex.Z(_as_topology(aVertex)))
        x_min = min(x)
        y_min = min(y)
        z_min = min(z)
        x_max = max(x)
        y_max = max(y)
        z_max = max(z)
        return [x_min, y_min, z_min, x_max, y_max, z_max]
    
    def is_intersecting(box1, box2):
        """
        Checks if two bounding boxes intersect.
        
        Parameters:
        - box1: List of [x_min, y_min, z_min, x_max, y_max, z_max] for the first box.
        - box2: List of [x_min, y_min, z_min, x_max, y_max, z_max] for the second box.
        
        Returns:
        - True if the boxes intersect, False otherwise.
        """
        return (box1[0] <= box2[3] and box1[3] >= box2[0] and  # Check X-axis overlap
                box1[1] <= box2[4] and box1[4] >= box2[1] and  # Check Y-axis overlap
                box1[2] <= box2[5] and box1[5] >= box2[2])      # Check Z-axis overlap
    
    def nested_list_to_dataframe(nested_list):
        # Extract the first sublist as column names
        columns = nested_list[0]
        # The rest of the sublists are the rows of data
        data = nested_list[1:]
        # Create the DataFrame
        df = pd.DataFrame(data, columns=columns)
        return df
    topologies = Topology.ByIFCFile(ifc_file, transferDictionaries=True)
    topologies = [t for t in topologies if Topology.IsInstance(_as_topology(t), 'Topology')]
    #Topology.Show(topologies)
    n = len(topologies)
    used = np.zeros((n, n))
    bb_list = []
    # Create a list of bounding boxes for quick intersection tests.
    for topology in topologies:
        bb_list.append(bb(topology))

    intersections = []
    df_list = [["GUID A", "GUID B", "X MIN A", "Y MIN A", "Z MIN A", "X MAX A", "Y MAX A", "Z MAX A", "X MIN B", "Y MIN B", "Z MIN B", "X MAX B", "Y MAX B", "Z MAX B"]]
    for i in range(0,n):
        for j in range(0, n):
            if not i == j and not used[i][j]:
                if is_intersecting(bb_list[i], bb_list[j]):
                    inter = (Topology.Intersect(topologies[i], topologies[j]))
                    if not inter == None:
                        intersections.append(inter)
                        d_i = _record_dictionary(topologies[i])
                        guid_i = Dictionary.ValueAtKey(d_i, "IFC_global_id")
                        d_j = _record_dictionary(topologies[j])
                        guid_j = Dictionary.ValueAtKey(d_j, "IFC_global_id")
                        row = [guid_i, guid_j]+bb_list[i]+bb_list[j]
                        df_list.append(row)
                used[i][j] = 1
                used[j][i] = 1
    figure = None
    plotly_figure = None
    if len(intersections) > 0:
        if figure == True:
            intersection_cluster = Cluster.ByTopologies(_as_topology(intersections))
            data01 = Plotly.DataByTopology(intersection_cluster, faceColor="red", faceOpacity=0.4, edgeColor="red", vertexColor="red", edgeWidth=3)
            data02 = Plotly.DataByTopology(Cluster.ByTopologies(_as_topology(topologies)), showEdges=False, showVertices=False, faceOpacity=0.2)
            plotly_figure = Plotly.FigureByData(data01+data02)
    else:
        if figure == True:
            data02 = Plotly.DataByTopology(Cluster.ByTopologies(_as_topology(topologies)), showEdges=False, showVertices=False, faceOpacity=0.2)
            plotly_figure = Plotly.FigureByData(data02)
    df = None
    if dataframe == True:
        df = nested_list_to_dataframe(df_list)
    return plotly_figure, df

# Code to run the function:
import pandas as pd
import ifcopenshell
ifc_file_path = r"C:\Users\sarwj\OneDrive - Cardiff University\Odilo\190056_31_BING_KUBA_WTK_structure.ifc"
ifc_file = ifcopenshell.open(ifc_file_path)
plotly_figure, df = get_intersections(ifc_file, dataframe=True, figure=True)
if isinstance(df, pd.core.frame.DataFrame):
    print(df)
if plotly_figure:
    Plotly.Show(plotly_figure)

In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.TGraph import TGraph
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary

c = CellComplex.Prism(width=4, length=4, height=4, uSides=4, vSides=4, wSides=4)
c1 = Cell.Prism()
c = Topology.Impose(c, c1)
c1 = Cell.Prism(width=2.5, length=2.5, height=2.5)
c = Topology.Merge(c, c1)
#Topology.Show(c)
g = TGraph.ByTopology(c)
mst = TGraph.MinimumSpanningTree(g)
#Topology.Show(mst)
# vertices = Topology.Vertices(c)
# edges = Topology.Edges(c)
# l_g = Graph.ByVerticesEdges(vertices, edges)
# Topology.Show(l_g)
# vertices = Graph.Vertices(l_g)
# print(len(vertices), "vertices")
# #g = Graph.ByTopology(c, toExteriorTopologies=True)
# #l_g = Graph.LineGraph(g, transferVertexDictionaries=False, transferEdgeDictionaries=True, silent=True)


# # ['circle 2D', 'grid 2D', 'line 2D', 'radial 2D', 'spring 2D', 'tree 2D', 'grid 3D', 'sphere 3D', 'tree 3D']
# mst = Graph.MinimumSpanningTree(l_g)
# connectivities =  Graph.Connectivity(mst)
# vertices = Graph.Vertices(mst)
# root_vertex = vertices[0]
# d = Topology.Dictionary(root_vertex)
# max_connectivity = Dictionary.ValueAtKey(d, "connectivity")
# for v in vertices:
#     d = Topology.Dictionary(v)
#     connectivity = Dictionary.ValueAtKey(d, "connectivity")
#     if connectivity > max_connectivity:
#         max_connectivity = connectivity
#         root_vertex = v
#     connectivity = connectivity*3 + 4
#     d = Dictionary.SetValueAtKey(d, "new_connectivity", connectivity)
#     v = Topology.SetDictionary(v, d)

# flat_g = Graph.Reshape(mst, shape="dendrimer 2D", rootVertex=root_vertex)
# #flat_g = Graph.Reshape(l_g, shape="line 2D")
# Topology.Show(flat_g, sagitta=0.05, absolute=False, vertexSizeKey="new_connectivity", vertexGroupKey="connectivity", vertexGroups=list(range(10)), vertexLabelKey="connectivity", colorScale="rainbow", width=1024, height=900)


In [ ]:
connectivities = TGraph.Connectivity(mst)
root_vertex = TGraph.NearestVertex(mst, Vertex.Origin())
flat_g = TGraph.Reshape(mst, shape="dendrimer 2D", rootVertex=root_vertex, size=1)
Topology.Show(c, flat_g)
flat_g = TGraph.Reshape(mst, shape="dendrimer 2D", rootVertex=root_vertex, size=100)
Topology.Show(c, flat_g)

# Topology.Show(flat_g, sagitta=0.1, absolute=False, vertexSize=12)
# Topology.Show(flat_g, sagitta=0.1, absolute=False, vertexSize=12, vertexGroupKey="connectivity", vertexGroups=list(range(6)))

# Topology.Show(mst, sagitta=0.1, absolute=False, vertexSize=12, vertexGroupKey="connectivity", vertexGroups=list(range(6)))
# Topology.Show(c, faceOpacity=0.2, vertexSize=3)
# leaves = Graph.Leaves(flat_g)
# Topology.Show(leaves, vertexSize=12)

In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

from topologicpy.TGraph import TGraph
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper

def find_communities(graph):
    """
    Finds communities of vertices in a TopologicPy graph.

    Args:
        graph (Graph): A TopologicPy graph instance.

    Returns:
        list: A list of communities, where each community is a list of vertex identifiers.
    """
    def dfs(vertex, visited, community):
        """Depth-First Search to explore the graph."""
        d = _record_dictionary(vertex)
        id = Dictionary.ValueAtKey(d, "_id_")
        visited.add(id)
        community.append(vertex)
        neighbors = TGraph.AdjacentVertices(graph, vertex)
        for neighbor in neighbors:
            d = _record_dictionary(neighbor)
            id = Dictionary.ValueAtKey(d, "_id_")
            if id not in visited:
                dfs(neighbor, visited, community)

    visited = set()
    communities = []
    vertices = TGraph.Vertices(graph)

    for vertex in vertices:
        d = _record_dictionary(vertex)
        id = Dictionary.ValueAtKey(d, "_id_")
        if id not in visited:
            community = []
            dfs(vertex, visited, community)
            communities.append(community)

    return communities

# Example Usage
# Assuming `my_graph` is a TopologicPy graph instance:
# communities = find_communities_topologicpy(my_graph)
# print(communities)

# def find_communities(graph, py_graph):
#     """
#     Finds communities of vertices in an undirected graph.
    
#     Args:
#         graph (dict): A dictionary where keys are vertex identifiers and values are lists of adjacent vertices.
        
#     Returns:
#         list: A list of communities, where each community is a list of vertices.
#     """
#     def dfs(py_vertex, tp_vertex, visited, py_community, tp_community):
#         """Depth-First Search to explore the graph."""
#         visited.add(py_vertex)
#         py_community.append(py_vertex)
#         tp_community.append(tp_vertex)
#         for py_neighbor in graph.get(vertex, []):
#             if neighbor not in visited:
#                 dfs(neighbor, visited, community)

#     visited = set()
#     communities = []
#     return_communities = []

#     for vertex in graph:
#         if vertex not in visited:
#             community = []
#             dfs(vertex, visited, community)
#             communities.append(community)

#     return communities

# Example Usage
# graph = {
#     'A': ['B', 'C'],
#     'B': ['A', 'D'],
#     'C': ['A'],
#     'D': ['B'],
#     'E': ['F'],
#     'F': ['E'],
#     'G': []
# }

c1 = CellComplex.Prism(uSides=3, vSides=3, wSides=3)
c2 = Topology.Translate(_as_topology(c1), 2, 0, 0)
c3 = Topology.Translate(_as_topology(c1), 2, 2, 0)
c4 = Topology.Translate(_as_topology(c1), 0, 2, 0)
t = Cell.Prism(origin=Vertex.ByCoordinates(1,1,0), width=1.25, length=1.25)
cluster = Cluster.ByTopologies(_as_topology(t), _as_topology(c1), _as_topology(c2), _as_topology(c3), _as_topology(c4))
cells = Topology.Cells(cluster)
final = CellComplex.ByCells(cells)
print(final)
Topology.Show(final)
graph = TGraph.ByTopology(final)
graph = TGraph.MinimumSpanningTree(graph)
vertices = TGraph.Vertices(graph)  # Get all vertices of the graph
for i, v in enumerate(vertices):
    d = _record_dictionary(v)
    d = Dictionary.SetValueAtKey(d, "_id_", i+1)
    v = Topology.SetDictionary(v, d)
communities = find_communities(graph)
print(len(communities))
for i, community in enumerate(communities):
    print(i)
    for v in community:
        d = _record_dictionary(v)
        d = Dictionary.SetValueAtKey(d, "community", i)
        v = Topology.SetDictionary(v, d)

Topology.Show(graph, vertexSize=10, vertexGroupKey="community", vertexGroups=list(range(len(communities))))



In [ ]:
import random

# Helper function to calculate modularity
def modularity(graph, communities):
    m = sum(len(neighbors) for neighbors in graph.values()) / 2  # Total number of edges
    modularity = 0.0
    # Calculate the modularity score based on current community structure
    for node, neighbors in graph.items():
        for neighbor in neighbors:
            if communities[node] == communities[neighbor]:
                modularity += 1
    modularity /= (2 * m)
    
    # Adjusting modularity to account for communities' internal connections
    community_modularity = {}
    for community in set(communities.values()):
        community_modularity[community] = 0
        for node, neighbors in graph.items():
            if communities[node] == community:
                for neighbor in neighbors:
                    if communities[neighbor] == community:
                        community_modularity[community] += 1
    
    community_modularity_sum = sum(val / (2 * m) for val in community_modularity.values())
    return modularity - community_modularity_sum

# Function to perform Louvain community detection
def louvain(graph):
    # Step 1: Initialize each node as its own community
    communities = {node: node for node in graph}
    modularity_prev = -1
    improvement = True
    
    while improvement:
        improvement = False
        # Step 2: Optimize modularity
        for node in graph:
            best_community = communities[node]
            best_modularity = modularity_prev
            best_community_neighbors = None
            
            # Consider the neighbors' communities
            for neighbor in graph[node]:
                temp_community = communities[neighbor]
                communities[node] = temp_community  # Assign node to neighbor's community
                
                # Calculate modularity for the new community assignment
                mod = modularity(graph, communities)
                if mod > best_modularity:
                    best_modularity = mod
                    best_community = temp_community
                    best_community_neighbors = neighbor
            
            # If modularity increased, move node to the best community
            if best_community != communities[node]:
                communities[node] = best_community
                improvement = True
                modularity_prev = best_modularity
        
        # Step 3: Create a new graph with the communities as nodes
        if improvement:
            new_graph = {}
            for node, community in communities.items():
                if community not in new_graph:
                    new_graph[community] = []
                print(node)
                print(graph.keys())
                for neighbor in graph[node]:
                    if communities[neighbor] != community:
                        new_graph[community].append(communities[neighbor])
            graph = new_graph
    
    return communities


graph = TGraph.AdjacencyDictionary(g)
print(graph)
communities = louvain(graph)
print(communities)


In [ ]:
import matplotlib.pyplot as plt

def draw_tree_xy(graph, root=None):
    """
    Draws a tree-like structure of a graph in the XY plane using pure Python.

    Parameters:
        graph (dict): A dictionary representing the graph with two keys:
                      - "vertices": A list of unique vertex identifiers.
                      - "edges": A list of tuples representing edges (start, end).
        root (optional): The root vertex to start the tree layout. If None, the first vertex is used.

    Returns:
        None
    """
    vertices = graph.get("vertices", [])
    edges = graph.get("edges", [])
    
    if not vertices or not edges:
        print("Graph must have vertices and edges.")
        return

    # Create adjacency list
    adjacency_list = {v: [] for v in vertices}
    for edge in edges:
        adjacency_list[edge[0]].append(edge[1])
        adjacency_list[edge[1]].append(edge[0])
    
    if root is None:
        root = vertices[0]
    
    # Perform a DFS to assign positions
    positions = {}
    visited = set()
    level_width = {}  # To track how many nodes are on each level
    
    def dfs(node, depth=0, x_offset=0):
        if depth not in level_width:
            level_width[depth] = 0
        
        # Assign position
        x = x_offset + level_width[depth]
        y = -depth
        positions[node] = (x, y)
        level_width[depth] += 1
        visited.add(node)
        
        # Visit neighbors
        for neighbor in adjacency_list[node]:
            if neighbor not in visited:
                dfs(neighbor, depth + 1, x)
    
    dfs(root)
    
    # Plot the graph
    plt.figure(figsize=(10, 6))
    for edge in edges:
        x_coords = [positions[edge[0]][0], positions[edge[1]][0]]
        y_coords = [positions[edge[0]][1], positions[edge[1]][1]]
        plt.plot(x_coords, y_coords, color="black", zorder=1)
    
    for vertex, (x, y) in positions.items():
        plt.scatter(x, y, color="lightblue", s=500, zorder=2)
        plt.text(x, y, str(vertex), ha="center", va="center", fontsize=10, zorder=3)
    
    plt.title("Tree Layout in XY Plane (Pure Python)")
    plt.axis("off")
    plt.show()

# Example Usage
graph = {
    "vertices": [1, 2, 3, 4, 5, 6],
    "edges": [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6)]
}

draw_tree_xy(graph, root=1)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_dendrimer_layout(graph):
    """
    Given a graph as an adjacency dictionary, this function generates a dendrimer layout
    and plots it using matplotlib.
    
    :param graph: dict, adjacency dictionary where keys are node ids and values are sets/lists of neighboring nodes
    """
    # Determine the layout of the graph
    positions = {}
    visited = set()
    layers = {}
    
    # Helper function to perform a DFS and organize nodes in layers
    def dfs(node, depth):
        visited.add(node)
        if depth not in layers:
            layers[depth] = []
        layers[depth].append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor, depth + 1)
    
    # Start DFS from an arbitrary node (e.g., node with key 0)
    starting_node = list(graph.keys())[0]
    dfs(starting_node, 0)
    
    # Compute positions based on layers
    for depth, nodes in layers.items():
        # Place nodes in a circular arrangement at each layer
        num_nodes = len(nodes)
        angle_step = 2 * np.pi / num_nodes
        for i, node in enumerate(nodes):
            angle = i * angle_step
            x = depth * np.cos(angle)
            y = depth * np.sin(angle)
            positions[node] = (x, y)
    
    # Plot the graph
    plt.figure(figsize=(8, 8))
    for node, (x, y) in positions.items():
        plt.scatter(x, y, s=100, c='blue', marker='o', zorder=5)
        plt.text(x, y, str(node), fontsize=12, ha='center', va='center', zorder=10)
    
    # Plot the edges
    for node, neighbors in graph.items():
        for neighbor in neighbors:
            if neighbor > node:  # Avoid double-counting edges
                x1, y1 = positions[node]
                x2, y2 = positions[neighbor]
                plt.plot([x1, x2], [y1, y2], c='black', lw=1, zorder=1)
    
    plt.axis('equal')
    plt.axis('off')
    plt.show()

# Example usage:
graph = {
    0: [1, 2, 3],         # Node 0 connected to nodes 1, 2, and 3
    1: [0, 4, 5, 6],      # Node 1 connected to nodes 0, 4, 5, and 6
    2: [0, 7, 8],         # Node 2 connected to nodes 0, 7, and 8
    3: [0, 9, 10],        # Node 3 connected to nodes 0, 9, and 10
    4: [1, 11, 12],       # Node 4 connected to nodes 1, 11, and 12
    5: [1, 13, 14],       # Node 5 connected to nodes 1, 13, and 14
    6: [1, 15, 16],       # Node 6 connected to nodes 1, 15, and 16
    7: [2, 17, 18],       # Node 7 connected to nodes 2, 17, and 18
    8: [2, 19, 20],       # Node 8 connected to nodes 2, 19, and 20
    9: [3, 21, 22],       # Node 9 connected to nodes 3, 21, and 22
    10: [3, 23, 24],      # Node 10 connected to nodes 3, 23, and 24
    11: [4, 25, 26],      # Node 11 connected to nodes 4, 25, and 26
    12: [4, 27, 28],      # Node 12 connected to nodes 4, 27, and 28
    13: [5, 29, 30],      # Node 13 connected to nodes 5, 29, and 30
    14: [5, 31, 32],      # Node 14 connected to nodes 5, 31, and 32
    15: [6, 33, 34],      # Node 15 connected to nodes 6, 33, and 34
    16: [6, 35, 36],      # Node 16 connected to nodes 6, 35, and 36
    17: [7, 37, 38],      # Node 17 connected to nodes 7, 37, and 38
    18: [7, 39, 40],      # Node 18 connected to nodes 7, 39, and 40
    19: [8, 41, 42],      # Node 19 connected to nodes 8, 41, and 42
    20: [8, 43, 44],      # Node 20 connected to nodes 8, 43, and 44
    21: [9, 45, 46],      # Node 21 connected to nodes 9, 45, and 46
    22: [9, 47, 48],      # Node 22 connected to nodes 9, 47, and 48
    23: [10],             # Node 23 connected to node 10
    24: [10],             # Node 24 connected to node 10
    25: [11],             # Node 25 connected to node 11
    26: [11],             # Node 26 connected to node 11
    27: [12],             # Node 27 connected to node 12
    28: [12],             # Node 28 connected to node 12
    29: [13],             # Node 29 connected to node 13
    30: [13],             # Node 30 connected to node 13
    31: [14],             # Node 31 connected to node 14
    32: [14],             # Node 32 connected to node 14
    33: [15],             # Node 33 connected to node 15
    34: [15],             # Node 34 connected to node 15
    35: [16],             # Node 35 connected to node 16
    36: [16],             # Node 36 connected to node 16
    37: [17],             # Node 37 connected to node 17
    38: [17],             # Node 38 connected to node 17
    39: [18],             # Node 39 connected to node 18
    40: [18],             # Node 40 connected to node 18
    41: [19],             # Node 41 connected to node 19
    42: [19],             # Node 42 connected to node 19
    43: [20],             # Node 43 connected to node 20
    44: [20],             # Node 44 connected to node 20
    45: [21],             # Node 45 connected to node 21
    46: [21],             # Node 46 connected to node 21
    47: [22],             # Node 47 connected to node 22
    48: [22],             # Node 48 connected to node 22
    49: [23, 24]          # Node 49 connected to nodes 23 and 24
}


plot_dendrimer_layout(graph)




In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import deque, defaultdict

def plot_hierarchical_layout(graph):
    """
    Given a graph as an adjacency dictionary, this function generates a hierarchical tree layout,
    center-justified, and plots it using matplotlib.
    
    :param graph: dict, adjacency dictionary where keys are node ids and values are sets/lists of neighboring nodes
    """
    # Step 1: Perform BFS/DFS to determine the layers of the tree.
    def bfs(root):
        layers = defaultdict(list)
        visited = set()
        queue = deque([(root, 0)])  # (node, depth)
        visited.add(root)

        while queue:
            node, depth = queue.popleft()
            layers[depth].append(node)
            
            for neighbor in graph.get(node, []):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, depth + 1))

        return layers

    # Start BFS from an arbitrary root node (node 0 in this case)
    root = list(graph.keys())[0]
    layers = bfs(root)
    
    # Step 2: Calculate positions for each node, center-justified
    positions = {}
    max_width = 0  # To track the maximum width of any layer for centering
    total_width = 0  # Total width to calculate global center

    # Step 3: Assign horizontal positions to nodes in each layer
    for depth, nodes in layers.items():
        num_nodes = len(nodes)
        angle_step = 1 / (num_nodes + 1)  # To center the nodes within the layer
        layer_width = 2 * num_nodes  # Space between nodes
        x_offset = (layer_width - num_nodes) / 2  # Center-justifying the nodes in the layer

        # Place nodes
        for i, node in enumerate(nodes):
            x = i + x_offset  # Horizontal position
            y = -depth  # Depth is negative to plot top-down
            positions[node] = (x, y)
            total_width = max(total_width, x + 1)

    # Step 4: Calculate the global offset to center the entire tree
    global_offset = total_width / 2

    # Adjust node positions to center the entire tree
    for node in positions:
        x, y = positions[node]
        positions[node] = (x - global_offset, y)

    # Step 5: Plotting the graph
    plt.figure(figsize=(12, 8))

    # Plot the edges
    for node, neighbors in graph.items():
        for neighbor in neighbors:
            if neighbor > node:  # Avoid duplicate edges (undirected graph)
                x1, y1 = positions[node]
                x2, y2 = positions[neighbor]
                plt.plot([x1, x2], [y1, y2], c='black', lw=1)

    # Plot the nodes
    for node, (x, y) in positions.items():
        plt.scatter(x, y, s=100, c='blue', marker='o', zorder=5)
        plt.text(x, y, str(node), fontsize=12, ha='center', va='center', zorder=10)

    # Set limits and appearance
    plt.axis('equal')
    plt.xlim(-total_width / 2, total_width / 2)
    plt.ylim(min(positions.values(), key=lambda x: x[1])[1] - 1, 1)
    plt.axis('off')
    plt.show()



plot_hierarchical_layout(graph)


In [ ]:
from collections import deque, defaultdict

def edge_betweenness_centrality(graph):
    betweenness = defaultdict(float)
    
    for s in graph:
        # Single-source shortest paths problem
        S = []
        P = defaultdict(list)
        sigma = dict.fromkeys(graph, 0.0)  # sigma[v]=0 for all v
        sigma[s] = 1.0
        D = dict.fromkeys(graph, -1)  # D[v]=-1 for all v
        D[s] = 0
        Q = deque([s])
        
        while Q:
            v = Q.popleft()
            S.append(v)
            for w in graph[v]:
                # Path discovery
                if D[w] < 0:  # w found for the first time?
                    Q.append(w)
                    D[w] = D[v] + 1
                # Path counting
                if D[w] == D[v] + 1:  # is (v, w) on the shortest path?
                    sigma[w] += sigma[v]
                    P[w].append(v)
        
        # Accumulation
        delta = dict.fromkeys(graph, 0.0)
        while S:
            w = S.pop()
            for v in P[w]:
                c = (sigma[v] / sigma[w]) * (1.0 + delta[w])
                if (v, w) not in betweenness and (w, v) not in betweenness:
                    betweenness[(v, w)] = 0.0
                betweenness[(v, w)] += c
                delta[v] += c
            if w != s:
                betweenness[(s, w)] += delta[w]
    
    # Normalize the betweenness values
    for edge in betweenness:
        betweenness[edge] /= 2.0
    
    return betweenness

# Example usage:
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'C', 'D'],
    'C': ['A', 'B', 'D'],
    'D': ['B', 'C']
}

betweenness = edge_betweenness_centrality(graph)
for edge, centrality in betweenness.items():
    print(f"Edge {edge}: {centrality:.2f}")


In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.TGraph import TGraph
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary

c = CellComplex.Prism(uSides=3, vSides=3)
g = TGraph.ByTopology(c, toExteriorTopologies=True)
Topology.Show(g)
new_g = TGraph.Complete(g)
Topology.Show(new_g)
# vertices = Graph.Vertices(g)
# edges = Graph.Edges(g)
# for i, edge in enumerate(edges):
#     d = Dictionary.ByKeyValue("id", i)
#     edge = Topology.SetDictionary(edge, d)

# for sv in vertices:
#     for ev in vertices:
#         if sv != ev:
#             edge = Graph.Edge(g, sv, ev)
#             print(edge)
#             s_edges = Graph.Edges(g, [sv])
#             e_edges = Graph.Edges(g, [ev])
#             for s_edge in s_edges:
#                 for e_edge in e_edges:
#                     if Edge.IsCollinear(s_edge, e_edge) and Edge.Length(s_edge) == Edge.Length(e_edge):
#                         s_d = Topology.Dictionary(s_edge)
#                         e_d = Topology.Dictionary(e_edge)
#                         s_value = Dictionary.ValueAtKey(s_d, "id")
#                         e_value = Dictionary.ValueAtKey(e_d, "id")
#                         s_d = Dictionary.SetValueAtKey(s_d, "id", 200)
#                         s_edge = Topology.SetDictionary(s_edge, s_d)
#                         print(s_value, " <> ", e_value)

# edges = Graph.Edges(g)
# for i, edge in enumerate(edges):
#     d = Topology.Dictionary(edge)
#     value = Dictionary.ValueAtKey(d, "id")
#     print(value)


In [ ]:


@staticmethod
def BetweenessCentrality(graph, key: str = "betweeness_centrality", mantissa: int = 6, tolerance: float = 0.001, silent: bool = False):
    """
        Returns the betweeness centrality measure of the input list of vertices within the input graph. The order of the returned list is the same as the order of the input list of vertices. If no vertices are specified, the betweeness centrality of all the vertices in the input graph is computed. See https://en.wikipedia.org/wiki/Betweenness_centrality.

    Parameters
    ----------
    graph : topologic_core.Graph
        The input graph.
    key : str , optional
        The dictionary key under which to save the betweeness centrality score. The default is "betweneess_centrality".
    mantissa : int , optional
        The desired length of the mantissa. The default is 6.
    tolerance : float , optional
        The desired tolerance. The default is 0.0001.

    Returns
    -------
    list
        The betweeness centrality of the input list of vertices within the input graph. The values are in the range 0 to 1.

    """
    def vertex_betweenness_centrality(graph, vertices):
        """
        Compute the betweenness centrality for vertices in the given TopologicPy graph.

        Args:
            graph: The TopologicPy Graph object.

        Returns:
            dict: A dictionary mapping each vertex to its betweenness centrality value.
        """
        from collections import defaultdict
        n = len(vertices)
        idList = []
        vertex_map = {}
        for i, v in enumerate(vertices):
            d = _record_dictionary(v)
            d = Dictionary.SetValueAtKey(d, "__id_", str(i))
            v = Topology.SetDictionary(v, d)
            idList.append(str(i))
            vertex_map[str(i)] = v
        if n < 2:
            return {v: 0.0 for v in idList}
        centrality = defaultdict(float)

        for source in idList:
            stack, paths, sigma = [], {}, {v: 0.0 for v in idList}
            sigma[source] = 1.0
            paths[source] = []

            queue = [source]
            while queue:
                v = queue.pop(0)
                stack.append(v)
                vertex = vertex_map[v]
                for neighbor in TGraph.AdjacentVertices(graph, vertex):
                    d = _record_dictionary(neighbor)
                    neighbor_id = Dictionary.ValueAtKey(d, "__id_")
                    if neighbor_id not in paths:
                        queue.append(neighbor_id)
                        paths[neighbor_id] = [v]
                    elif v not in paths[neighbor_id]:
                        paths[neighbor_id].append(v)
                    sigma[neighbor_id] += sigma[v]

            delta = {v: 0.0 for v in idList}
            while stack:
                w = stack.pop()
                for v in paths.get(w, []):
                    delta[v] += (sigma[v] / sigma[w]) * (1 + delta[w])
                if w != source:
                    centrality[w] += delta[w]
        # Normalize centrality values
        max_centrality = max([centrality[v] for v in idList])
        min_centrality = min([centrality[v] for v in idList])
        centrality = [round((centrality[v]-min_centrality)/max_centrality, mantissa) for v in idList]
        for i, v in enumerate(vertices):
            d = _record_dictionary(v)
            d = Dictionary.SetValueAtKey(d, "betweeness_centrality", centrality[i])
            d = Dictionary.RemoveKey(d, "__id_")
            v = Topology.SetDictionary(v, d)
        return centrality
    
    from topologicpy.Topology import Topology
    from topologicpy.Dictionary import Dictionary

    if not Topology.IsInstance(_as_topology(graph), 'Graph'):
        if not silent:
            print("Graph.BetweenessCentrality - Error: The input graph is not a valid graph. Returning None.")
        return None
    
    vertices = TGraph.Vertices(graph)

    if len(vertices) < 1:
        if not silent:
            print("Graph.BetweenessCentrality - Error: The input graph does not contain valid vertices. Returning None.")
        return None
    if len(vertices) == 1:
        d = _record_dictionary(vertices[0])
        d = Dictionary.SetValueAtKey(d, key, 1.0)
        vertices[0] = Topology.SetDictionary(vertices[0], d)
        return [1.0]
    
    return vertex_betweenness_centrality(graph, vertices)

c = CellComplex.Prism(uSides=2, vSides=2)
c2 = CellComplex.Prism(width=0.5, length=0.5, height=0.5)
c2 = Topology.Translate(_as_topology(c2), 0.25, 0.25, 0.25)
c = Topology.Impose(c, c2)
g = TGraph.ByTopology(c, toExteriorTopologies=True)
centrality = BetweenessCentrality(g, key = "betweeness_centrality", mantissa = 2, tolerance = 0.001, silent = False)
centrality = list(set(centrality))
centrality.sort()
print(centrality)
Topology.Show(g, vertexSize=12, vertexGroupKey="betweeness_centrality", vertexGroups=centrality, colorScale="thermal")
b_cen = TGraph.BetweennessCentrality(g, mantissa=2)
b_cen = list(set(b_cen))
b_cen.sort()
print(b_cen)
Topology.Show(g, vertexSize=12, vertexGroupKey="betweeness_centrality", vertexGroups=b_cen, colorScale="thermal")

In [ ]:

# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.TGraph import TGraph
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary


from topologicpy.Helper import Helper

c = CellComplex.Prism(uSides=4, vSides=4, wSides=1)
c_temp = Topology.Rotate(_as_topology(c), angle=45)
c = Topology.Merge(c, c_temp)
c0 = Cell.Prism(width=1.5, length=1.5)
c1 = Topology.Translate(_as_topology(c), -1, -1, 0)
c2 = Topology.Translate(_as_topology(c), 1, -1, 0)
c3 = Topology.Translate(_as_topology(c), 1, 1, 0)
c4 = Topology.Translate(_as_topology(c), -1, 1, 0)
cells = [Topology.Cells(x) for x in [c1, c2, c3, c4]]
cells.append(c0)
cells = Helper.Flatten(cells)
cellComplex = CellComplex.ByCells(cells)


In [ ]:
g = TGraph.ByTopology(cellComplex, direct=False, viaSharedTopologies=True)
Topology.Show(g)
nx_graph = TGraph.NetworkXGraph(g)
g = TGraph.ByNetworkXGraph(nx_graph)
Topology.Show(g)
communities = TGraph.Community(g)
communities = list(set(communities))
communities.sort()
print(communities)
Topology.Show(g, vertexGroupKey="community", vertexGroups=communities, vertexSize=12, vertexLabelKey="community")

In [ ]:
from topologicpy.TGraph import TGraph
from topologicpy.Vertex import Vertex
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology

#def Community(graph, key="community", edgeWeightKey=None):
def louvain_method(vertices, edges, edge_weights):
    from collections import defaultdict

    def modularity_gain(node, community, graph, total_weight, weights):
        """Calculate modularity gain when moving a node to a community."""
        k_i_in = sum(weights[edge] for edge in graph[node] if edge in community)
        k_i = sum(weights[edge] for edge in graph[node])
        sum_tot = sum(sum(weights[edge] for edge in graph[neighbor]) for neighbor in community)
        return (k_i_in / total_weight) - (k_i / total_weight) * (sum_tot / total_weight)

    def initialize_communities(vertices):
        """Initialize each vertex in its own community."""
        return {vertex: vertex for vertex in vertices}

    def build_graph(vertices, edges):
        """Create a graph structure for fast access."""
        graph = defaultdict(list)
        for idx, (u, v) in enumerate(edges):
            graph[u].append(v)
            graph[v].append(u)
        return graph

    def aggregate_graph(communities, graph, weights):
        """Aggregate the graph based on current communities."""
        new_graph = defaultdict(list)
        new_weights = defaultdict(float)
        for (u, v), weight in weights.items():
            new_u, new_v = communities[u], communities[v]
            if new_u != new_v:
                new_graph[new_u].append(new_v)
                new_graph[new_v].append(new_u)
                new_weights[(new_u, new_v)] += weight
        return new_graph, new_weights

    # Convert edge weights to a dictionary for easy access
    weights = {edge: edge_weights[idx] for idx, edge in enumerate(edges)}
    total_weight = sum(edge_weights)

    graph = build_graph(vertices, edges)
    communities = initialize_communities(vertices)

    while True:
        improved = False
        for node in vertices:
            current_community = communities[node]
            best_community = current_community
            best_gain = 0

            # Test moving to each neighbor's community
            for neighbor in graph[node]:
                neighbor_community = communities[neighbor]
                if neighbor_community == current_community:
                    continue

                # Calculate modularity gain
                gain = modularity_gain(node, [n for n in communities if communities[n] == neighbor_community],
                                    graph, total_weight, weights)

                if gain > best_gain:
                    best_community = neighbor_community
                    best_gain = gain

            if best_community != current_community:
                communities[node] = best_community
                improved = True

        if not improved:
            break

        # Aggregate graph and repeat
        graph, weights = aggregate_graph(communities, graph, weights)
        vertices = list(set(communities.values()))

    # Map community IDs to output
    community_ids = list(set(communities.values()))
    community_map = {community: idx for idx, community in enumerate(community_ids)}
    return [community_map[communities[node]] for node in range(len(vertices))]

# Example usage
vertices = [0, 1, 2, 3, 4]
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)]
edge_weights = [1, 1, 1, 1, 1]

community_ids = louvain_method(vertices, edges, edge_weights)
print("Community IDs:", community_ids)


#     graph_edges = Graph.Edges(graph)
#     if edgeWeightKey == None:
#         weights = [1]*len(graph_edges)
#     else:
#         for edge in graph_edges:
#             d = Topology.Dictionary(edge)
#             weight = Dictionary.ValueAtKey(d, edgeWeightKey) or 1
#             weights.append(weight)

#     print(weights)
#     vertices = list(range(len(Graph.Vertices(graph))))
#     print("Vertices:", vertices)
#     mesh_g = Graph.MeshData(graph)
#     edges = mesh_g['edges']
#     print(edges)
#     communities = louvain_community_detection(vertices, edges, weights)
#     print("Communities:", communities)
#     return communities

# communities = Community(g)
# print(communities)


In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
from topologicpy.Vertex import Vertex
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph
from topologicpy.Helper import Helper
from topologicpy.Cluster import Cluster


components = TGraph.ConnectedComponents(graph1)
# Show the three largest components
Topology.Show(components,
            nameKey="IFC_name",
            sagitta= 0.05,
            absolute=False,
            faceOpacity=0.1,
            vertexSizeKey="closeness_centrality",
            vertexLabelKey="IFC_name",
            vertexGroupKey="IFC_type",
            vertexGroups=["Unknown", "IfcSpace", "IfcSlab", "IfcRoof", "IfcWall", "IfcWallStandardCase", "IfcDoor", "IfcWindow"],
            showVertexLegend = False,
            showEdgeLegend = False,
            showFaceLegend = False,
            backgroundColor="white", width=1024, height=900)

In [ ]:
import pandas as pd

def dicts_to_dataframe(dicts):
    """
    Converts a list of dictionaries into a pandas DataFrame, handling scalar values
    and optional filtering by keys.

    Parameters:
        dicts (list of dict): A list of dictionaries to convert.
        keys_to_include (list, optional): A list of keys to include in the DataFrame. 
                                          If None, all keys are included.

    Returns:
        pd.DataFrame: A DataFrame representation of the input dictionaries.
                      Missing values are filled with NaN.
    """
    try:
        # Create DataFrame
        df = pd.DataFrame(dicts)
        return df
    except Exception as e:
        print(f"Error: {e}")
        print("Ensure that all inputs are dictionaries with lists or scalars as values.")
        return None

def filter_dicts_by_keys(dicts, keys_to_include):
    """
    Filters a list of dictionaries to include only specified keys.

    Parameters:
        dicts (list of dict): A list of dictionaries to filter.
        keys_to_include (list of str): A list of keys to retain in each dictionary.

    Returns:
        list of dict: A new list of dictionaries containing only the specified keys.
    """
    if not isinstance(dicts, list):
        raise ValueError("Input must be a list of dictionaries.")
    
    filtered_dicts = []
    for d in dicts:
        if not isinstance(d, dict):
            raise ValueError("Each element in the list must be a dictionary.")
        filtered_dicts.append({k: v for k, v in d.items() if k in keys_to_include})
    
    return filtered_dicts


vertices = TGraph.Vertices(graph1)
dictionaries = [_record_dictionary(v) for v in vertices]
py_dicts = [Dictionary.PythonDictionary(d) for d in dictionaries]
# Keys to include
keys_to_include = ["IFC_global_id", "IFC_name", "IFC_type"]
filtered_dicts = []
# Filter the data
filtered_dicts = filter_dicts_by_keys(py_dicts, keys_to_include)
print(filtered_dicts)
df = dicts_to_dataframe(filtered_dicts)
print(df)

In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Topology import Topology



# Create an L-shaped polygon with specific parameters
i_shape_f = Face.TShape(width=0.5, a=0.25, b=0.5, placement="lowerleft")
#i_shape_f = Face.ByWire(l_shape)
n = Face.NormalEdge(i_shape_f)
# Visualize the shape using Plotly
Topology.Show(i_shape_f, n, xAxis=True, yAxis=True, vertexSize=3)
